In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import tensorflow as tf
#from tensorflow.keras import layers, models, losses
#from tensorflow.keras.callbacks import ModelCheckpoint
from keras import layers, models, losses, regularizers
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

I0000 00:00:1783068869.987517    5374 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print("GPU Trovate:", len(tf.config.list_physical_devices('GPU')))
for gpu in tf.config.list_physical_devices('GPU'):
    print("Nome:", gpu.name)

# 1. SETUP MEMORIA: DEVE ESSERE LA PRIMA COSA IN ASSOLUTO
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memoria GPU configurata in modalità dinamica.")
    except RuntimeError as e:
        print("Errore GPU:", e)

# 2. ESORCISMO DELLA RAM (Uccide i vecchi modelli in memoria)
tf.keras.backend.clear_session()

GPU Trovate: 1
Nome: /physical_device:GPU:0
Memoria GPU configurata in modalità dinamica.


# [LOG] Model Versioning

## Version 1: Tiny-Baseline (Obsolete)
**Data:** 15/05/2026
**Fase:** Upper-Bound Baseline (Test di fattibilità hardware)

### Architettura:
- **Input:** (1, 120, 18) -> [H, W, Channels]
- **Feature Extraction:** - Conv2D (16 filtri, kernel 1x5) + MaxPooling (1x2)
    - SeparableConv2D (32 filtri, kernel 1x3) + MaxPooling (1x2)
- **Output Heads:** - `coords_head`: Dense(8) [Linear] -> X, Y per 4 persone.
    - `mask_head`: Dense(4) [Sigmoid] -> Presenza per 4 persone.

### Statistiche:
- **Parametri Totali:** ~3,500
- **Peso Modello (Float32):** 13.67 KB
- **Peso Stimato (INT8 Quantized):** ~3.5 KB
- **Performance (Epoca 15):** - `val_loss`: 3.79
    - `val_coords_loss`: 3.35 (Errore spaziale medio ~1.83m)
    - `val_mask_loss`: 0.88

## Version 2: Capacità Espansa (Obsolete)
* **Performance:** Errore medio ~1.69m. 
* **Note:** La rete ha smesso di imparare dopo 37 epoche. Mancanza di regolarizzazione (Dropout) e LR fisso.

## Version 3: Architettura "Romana" (Heavy + Callbacks)
**Data:** 31/05/2026
**Fase:** Ottimizzazione Avanzata
* **Architettura:** 12 Layer. Doppie Conv2D(32) -> Doppie SepConv2D(64) -> Conv2D(128) -> Dense(128) + Dropout(0.3) -> Dense(64).
* **Data Pipeline:** `alpha=0.20` (EMA decluttering veloce, come da specifiche). Split dataset con casi complessi (3/4 persone) nel Training. `Batch_Size=8`.
* **Training Hacks:** - `ReduceLROnPlateau`: dimezza il learning rate se la loss si blocca.
    - `EarlyStopping`: ferma l'addestramento se non migliora per 10 epoche e ricarica i pesi migliori.
    - Metrica `RootMeanSquaredError`: legge l'errore spaziale direttamente in Metri.
* **Performance Spaziale:** L'errore medio sulle coordinate è sceso al minimo storico di **~1.56m**.

## [LA SERIE DEI DISASTRI TEORICI: V4 - V6]
*I modelli seguenti rappresentano tentativi falliti di risolvere il problema dell'assegnazione spaziale (Permutation Invariance) e di ignorare i "fantasmi" (persone non presenti). Hanno portato a una serie di Mode Collapse a causa di bug matematici e concettuali.*

## Version 4: Architettura "Imperiale" (Obsolete - Disastroso / Mode Collapse)

### Modifiche Apportate (La Teoria):
- **Spatial Sorting:** Ordinamento forzato dei target da sinistra a destra sull'asse X nel DataGenerator per fornire una regola fissa alla rete (aggirare la *permutation invariance*). **GRAVE ERRORE!!**
- **True Masked MSE:** Modifica della Loss function per moltiplicare l'errore per 0 quando la persona non è presente, smettendo di penalizzare la rete per i "fantasmi".
- **Bilanciamento Loss:** Il peso della `mask_head` è stato portato da 0.5 a 5.0 per costringere l'ottimizzatore a prestare attenzione alla presenza.
- **Custom Metric:** Creata `true_masked_rmse_metres` per calcolare correttamente l'errore in metri gestendo gli array concatenati a 12 valori (8 coords + 4 maschere).

### Statistiche e Limiti Hardware:
- Aumentare indiscriminatamente i filtri a 128 e i layer Densi a 256 ha causato un **esplosione della memoria Flash stimata a 927.42 KB**, superando il limite tassativo di 800 KB dell'ESP32. 

### Il Disastro (Performance):
- Rispetto al modello V3 (e allo split di Davide), **la resa visiva è pessima**. 
- **Sintomo:** Nel visualizzatore, le predizioni (le X rosse) non inseguono minimamente i bersagli. Rimangono immobili, raggruppate e appiccicate in un singolo punto nell'angolo in basso a sinistra della stanza.
- **Causa:** Il layer `GlobalAveragePooling2D`. Calcolando la media matematica su tutta l'ultima mappa di estrazione, ha letteralmente distrutto ogni informazione geometrica. La rete è diventata **cieca**. Non sapendo *dove* guardare, ha applicato un "Mode Collapse": ha imparato a sparare tutte le previsioni nel punto medio statistico per subire la minor penalità possibile dalla Loss.

## Version 5: Architettura "Occhiali" (Obsolete - Mode Collapse Persistente)
* **Modifica Architetturale:** Sostituito il `GlobalAveragePooling2D` con il layer `Flatten()` per ridare alla rete la "vista" geometrica. Usati `MaxPooling2D` aggressivi per mantenere la Flash stimata sotto gli 800 KB (scesa a ~223 KB).
* **Risultato:** Fallimento. Le predizioni continuano a non seguire i bersagli.
* **Diagnosi (Il vero colpevole):** Il problema non era solo il Pooling, ma lo **Spatial Sorting** introdotto nella V4. Poiché le persone si incrociano nella stanza, ordinare le coordinate sull'asse X frame per frame causava il "teletrasporto" dei target da un output all'altro. La rete, non avendo memoria temporale (LSTM), non riusciva a gestire questi sbalzi di gradiente e collassava statisticamente.

## Version 6: Architettura "Tracciante Naturale" (Obsolete - Bug Matematico)
* **Modifica:** Rimosso lo Spatial Sorting, ripristinato l'ordine naturale del dataset. Fixato il bug `axis=1` nella Loss.
* **Risultato:** Fallimento totale e crollo della Binary Accuracy (~50%). Errore fisso a 1.54m misurato magicamente al centro della stanza.
* **Causa 1 (Bug Keras):** Nella funzione custom *True Masked MSE*, mancava il parametro `axis=1` nell'istruzione `tf.reduce_sum()`. Questo fondeva l'errore di un intero batch in un singolo scalare, distruggendo completamente i gradienti frame-per-frame e lobotomizzando la rete.
* **Causa 2:** Problema dell'Assegnazione. Senza ordinamento spaziale, la rete (che non ha memoria temporale LSTM) non sa in quale delle 4 teste di output piazzare le coordinate di una persona in un singolo frame isolato, finendo per sparare al centro statistico per minimizzare la penalità.

## Version 7: Architettura "Ungara" (Breakthrough)
**Fase:** Risoluzione del Problema di Assegnazione
* **Modifica Teorica:** Fusa l'architettura in un'unica testa di output da 12 valori. Introdotta la **Total Hungarian Loss** (Permutation Invariant Training): la rete ora calcola l'errore per tutte le 24 permutazioni possibili e impara solo dall'incrocio geometricamente perfetto, eliminando il Mode Collapse.
* **Risultato:** Le predizioni si sbloccano dal centro della stanza e iniziano a inseguire fisicamente i target reali in movimento.
* **Criticità rilevate (Bug di Misurazione e Metodo):**
    1. **Bug Metrica Spaziale:** Il calcolo dell'errore (0.95m riportati) divideva per il numero di assi, distorcendo la trigonometria (il vero errore euclideo era ~1.34m).
    2. **Bug Metrica Maschera:** Keras misurava l'accuratezza binaria (riportata al 44%) su tutti i 12 output, mescolando coordinate e probabilità.
    3. **Data Split Sbilanciato:** Il set di Validazione ometteva del tutto gli scenari con 2 persone, minando la validità scientifica del test.!!!!

## Version 8: Architettura "Corazzata" (The Truth Fix)

**Fase**: Rigore Matematico e Hard-Limit Hardware

* **Data Split Stratificato:** Copertura totale in Validation (0, 1, 2, 3 e 4 soggetti) per testare l'algoritmo su tutti gli scenari.

* **Architettura Espansa:** Filtri raddoppiati (fino a 128) e livello Dense portato a 256. Saturazione consapevole della Flash (~683 KB) per estrarre il massimo dettaglio dai radar. Unione definita delle teste (Concatenate) per sincronizzare il Permutation Invariant Training.

* **Fix Matematici:** Metrica Euclidea reale (hungarian_rmse_euclidean_metres) e metrica di accuratezza maschere purificata (hungarian_mask_acc).

* **Performance (Epoca 23 - Golden):**

    *val_hungarian_rmse_metres: ~1.07m (Distanza Reale Fisica).

    *val_hungarian_mask_acc: ~77% (Capacità di distinguere fantasmi da persone reali).

* **Risultato Visivo:** Le predizioni inseguono fedelmente i bersagli garantendo il requisito F1 Score (True Positive <= 1.0m). Permane un fisiologico sfarfallio sui "fantasmi" dovuto all'assenza di memoria temporale (LSTM).

## Version 8.1: Architettura "Corazzata" (Full-Batch Survival)
**Data:** 05/06/2026
**Fase:** Sblocco Hardware GPU, Diagnosi e Record Full-Batch

### Requisiti Hardware (ESP32-S3 - FLOAT32):
- **Memoria FLASH Stimata:** 683.92 KB (✅ Limite: 800 KB).
- **Memoria SRAM Stimata (Activation Arena):** ~8.44 KB (✅ Limite: 300 KB).


## Version 9: Fusion
* Aggiunta standardization


**VARI ERRORI RISCONTRATI**
1. Log-Scaling sul Segnale (Tentativo di salvare i segnali deboli)

* L'Idea Teorica: Applicare np.log1p(mag) prima del filtro EMA per schiacciare i picchi altissimi dei muri (clutter statico) ed elevare i micro-segnali delle persone ferme.
* Il Risultato (Fallimento): Keras ha interrotto l'addestramento all'Epoca 20 (Early Stopping). La Training Loss è crollata (0.33m), ma la Validation Loss è rimasta inchiodata (0.47m). Il visualizzatore mostrava "X" rosse instabili e tremolanti.
* L'Autopsia (La Causa): Overfitting indotto da rumore. Il logaritmo ha amplificato non solo le persone ferme, ma anche tutto il micro-rumore radio di fondo della stanza. La V8 Corazzata (avendo 128 filtri e molta capacità mnemonica) ha "imparato a memoria" la mappa del rumore del set di addestramento per barare sulla Loss, fallendo poi miseramente sui dati di validazione.
* Soluzione Adottata: Rollback ai dati lineari. Transizione alla Standardizzazione Globale ($\mu$ e $\sigma$) per stabilizzare i gradienti senza amplificare il rumore.

2. V9 Feature-Level Fusion (Separazione dei Radar con Reshape)

* L'Idea Teorica: Usare layers.Reshape((6, 120, 3)) per separare i 18 canali in 6 radar indipendenti, in modo da processarli con filtri (1, 5) senza mischiarli fin dall'inizio. L'obiettivo era risolvere l'effetto "Blob" (persone vicine fuse insieme).
* Il Risultato (Fallimento): La Loss non è scesa in modo significativo e il visualizzatore ha mostrato che l'effetto "Blob" tra persone vicine è rimasto identico. Inoltre, non ha risolto la cecità sulle persone ferme.
* L'Autopsia (La Causa): Schiacciamento della risoluzione spaziale. L'effetto Blob non era causato dall'Early Fusion, ma dai troppi layer MaxPooling2D. Riduciamo i range bins da 120 a 15 prima del layer Dense. Con una stanza lunga 7.2 metri, 15 bins significano mezzo metro per ogni pixel radar. Due persone a 40 cm di distanza collassano fisicamente nello stesso bin del tensore. Separare i radar a monte non serve a nulla se poi a valle comprimiamo la risoluzione spaziale a blocchi da 50 centimetri.
* Soluzione Adottata: Abbandono della V9. Ritorno imminente alla V8 (Early Fusion) ma con l'idea di ridurre il Pooling per riacquistare risoluzione.

eeai_best_model_v8.keras

### Modifiche Apportate (Ingegneria di Sistema):
- **Accensione Reattore GPU:** Risolto il blocco driver passando dall'addestramento su CPU a quello su NVIDIA GTX 1650 (4GB VRAM) tramite l'installazione nativa dei toolkit CUDA e cuDNN via Conda.
- **cuDNN Autotuner Bypass:** Disattivato l'autotuning di TensorFlow (`TF_CUDNN_USE_AUTOTUNE = 0`). L'Autotuner tentava un collaudo di algoritmi convoluzionali che richiedeva oltre 1GB di workspace VRAM, portando la scheda al crash istantaneo. Disattivandolo, l'addestramento procede in modo deterministico e sicuro.
- **Memoria Dinamica e OOM Fix:** Inserito esorcismo della RAM (`clear_session()`) per distruggere i grafi orfani. Impostato il limite tassativo `BATCH_SIZE = 1` nel `EEAIDataGenerator` per evitare il crash per "Indigestione" da allocazione VRAM singola (>2.5 GB).

### Performance:
- **`val_hungarian_rmse_metres`:** **0.8380m** (Nuovo record assoluto! Muro dell'1.0m ufficialmente abbattuto).
- **`val_hungarian_mask_acc`:** **79.23%** (Ottima reiezione dei fantasmi).
- **Tempo di addestramento:** ~15 secondi per Epoca.

### Il Limite Concettuale Scoperto:
- Sebbene la rete abbia performato benissimo, si è scoperta una grave anomalia architetturale nel DataGenerator legata al `BATCH_SIZE = 1`.
- Keras interpreta `1` non come *1 frame*, ma come *1 file intero* (circa 15.000 frame cronologici).
- **L'Errore Matematico:** Questo costringe la rete a eseguire un **Full-Batch Gradient Descent**. La rete fa la media degli errori di tutte le 15.000 istantanee e aggiorna i suoi pesi **una sola volta** per ogni file. Con 18 file di Train, la rete ha compiuto appena 18 passi di discesa del gradiente per ogni Epoca.

### Verdetto:
Questa versione rappresenta la **Baseline Assoluta (Strong)** dell'infrastruttura. Aver raggiunto 0.83m con soli 18 aggiornamenti per epoca dimostra che l'architettura V8 "Corazzata" e la funzione di Loss Ungherese sono perfette. Tuttavia, segna la fine dell'uso del `tf.keras.utils.Sequence` per questo dataset: la rete è pronta per il VERO Machine Learning. Passando all'infrastruttura in RAM (Mini-Batch da 32), la rete passerà da 18 aggiornamenti a oltre 6.000 aggiornamenti per epoca, aprendo le porte alla micro-cesellatura dei pesi.

# [GUIDA] Gerarchia del Fine-Tuning per Edge AI (ESP32-S3)

Nel TinyML non possiamo ingrandire la rete a caso, perché siamo limitati da 400KB di RAM e dalla latenza. Le modifiche seguono un ordine di priorità basato sul **Costo Hardware**.

### Livello 1: Costo Hardware ZERO (Modifiche di Addestramento)
Questi parametri non alterano il peso finale del file `.tflite`. Si provano per primi.
* **1. Epoche (`epochs`):** * *Cos'è:* Il tempo di studio. Quante volte la rete vede l'intero dataset.
    * *Quando usarlo:* Se la `val_loss` sta scendendo ma l'addestramento finisce troppo presto (Underfitting).
    * *Effetto:* Permette alla rete di continuare a correggere gli errori.
* **2. Learning Rate (`lr`):**
    * *Cos'è:* La "lunghezza del passo" durante la discesa del gradiente.
    * *Quando usarlo:* Se la Loss salta su e giù in modo impazzito (LR troppo alto) o se non scende per niente fin dall'inizio (LR troppo basso).
    * *Effetto:* Rende l'apprendimento più stabile o più aggressivo.

### Livello 2: Costo Hardware BASSO (Capacità / Larghezza)
* **3. Numero di Filtri (es. da 32 a 64):**
    * *Quando usarlo:* Se la rete è troppo "stupida" per capire le dinamiche della stanza e la Loss si blocca su valori alti (come nella nostra V1).
    * *Effetto:* Aumenta i parametri (Flash) e leggermente la RAM. Dà alla rete più "neuroni" per capire la trigonometria.

### Livello 3: Costo Hardware ALTO (Profondità / Latenza)
* **4. Aggiungere Layer (es. una terza Conv2D):**
    * *Cos'è:* Aggiungere step sequenziali al modello.
    * *Quando usarlo:* Solo se la rete larga non basta per estrarre concetti complessi.
    * *Effetto:* Aumenta drasticamente le operazioni matematiche (MACs). **Aumenta la latenza:** l'ESP32 ci metterà molto più tempo a calcolare ogni singolo frame. Usare con estrema cautela.

In [20]:
# ==============================================================================
# 1. DATA ENGINE V9 (Caricamento Globale in RAM)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        # Inizializza il background per l'EMA
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # L'EMA viene calcolato qui, una volta per tutte, in perfetto ordine cronologico!
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # Flatten delle coordinate e concatenazione con la mask (12 valori totali)
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pre-calcolati)")

    # Uniamo tutte le liste in due immensi tensori Numpy pronti per la GPU
    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)

    mean_val = np.mean(X)
    std_val = np.std(X)
    print(f"VALORI DA SALVARE PER CODE.PY -> MEAN: {mean_val:.4f}, STD: {std_val:.4f}")
    
    X = (X - mean_val) / (std_val + 1e-7)

    return X, Y

# ==============================================================================
# 2. SPLIT STRATIFICATO RIGOROSO
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 3. ESECUZIONE DEL MOTORE (ATTENZIONE: Ci metterà ~1 minuto a caricare tutto in RAM)
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato. (7500 frame pre-calcolati)
File 2/18 processato. (7500 frame pre-calcolati)
File 3/18 processato. (7500 frame pre-calcolati)
File 4/18 processato. (7500 frame pre-calcolati)
File 5/18 processato. (7500 frame pre-calcolati)
File 6/18 processato. (7500 frame pre-calcolati)
File 7/18 processato. (7500 frame pre-calcolati)
File 8/18 processato. (7500 frame pre-calcolati)
File 9/18 processato. (7500 frame pre-calcolati)
File 10/18 processato. (7500 frame pre-calcolati)
File 11/18 processato. (7500 frame pre-calcolati)
File 12/18 processato. (7500 frame pre-calcolati)
File 13/18 processato. (7500 frame pre-calcolati)
File 14/18 processato. (7500 frame pre-calcolati)
File 15/18 processato. (7500 frame pre-calcolati)
File 16/18 processato. (7500 frame pre-calcolati)
File 17/18 processato. (7500 frame pre-calcolati)
File 18/18 processato. (7500 frame pre-calcolati)
VALORI DA SALVARE PER

In [3]:
# ==============================================================================
# 1. DATA ENGINE V9 (Caricamento Globale in RAM - SENZA NORMALIZZAZIONE INTERNA)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.002):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        # Inizializza il background per l'EMA
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # EMA
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # Flatten delle coordinate e concatenazione con la mask (12 valori totali)
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    # Uniamo tutte le liste in due tensori Numpy
    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)

    return X, Y

# ==============================================================================
# 2. SPLIT E ESECUZIONE (NORMALIZZAZIONE RIGOROSA)
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)

print("\n--- CALCOLO STATISTICHE E NORMALIZZAZIONE ---")
# LE STATISTICHE SI CALCOLANO SOLO SUL TRAIN!
train_mean = np.mean(X_train_raw)
train_std = np.std(X_train_raw)

print(f"!!! VALORI DA SALVARE E HARDCODARE NEL TUO CODE.PY !!!")
print(f"MEAN: {train_mean:.6f}")
print(f"STD:  {train_std:.6f}")

# Applichiamo la stessa normalizzazione a entrambi
X_train = (X_train_raw - train_mean) / (train_std + 1e-7)
X_val = (X_val_raw - train_mean) / (train_std + 1e-7)

# Liberiamo memoria RAM
del X_train_raw
del X_val_raw


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato.
File 2/18 processato.
File 3/18 processato.
File 4/18 processato.
File 5/18 processato.
File 6/18 processato.
File 7/18 processato.
File 8/18 processato.
File 9/18 processato.
File 10/18 processato.
File 11/18 processato.
File 12/18 processato.
File 13/18 processato.
File 14/18 processato.
File 15/18 processato.
File 16/18 processato.
File 17/18 processato.
File 18/18 processato.

--- PREPARAZIONE VALIDATION SET ---
Inizio caricamento ed EMA Decluttering di 6 file...
File 1/6 processato.
File 2/6 processato.
File 3/6 processato.
File 4/6 processato.
File 5/6 processato.
File 6/6 processato.

--- CALCOLO STATISTICHE E NORMALIZZAZIONE ---
!!! VALORI DA SALVARE E HARDCODARE NEL TUO CODE.PY !!!
MEAN: 13.715407
STD:  45.612080


In [8]:
# ==============================================================================
# 1. DATA ENGINE V9 (Caricamento Globale in RAM, no normalizzazione interna - Shape: 6, 120, 3) 
# ==============================================================================


def load_and_process_all_files(file_list, alpha=0.002):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   # Shape: (T, 6, 3, 120, 2)
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        # Calcolo Magnitudo: (T, 6, 3, 120)
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        
        # TRUCCO MAGISTRALE: Trasponiamo per avere (T, Radars, Bins, Antennas) -> (T, 6, 120, 3)
        mag_reshaped = np.transpose(mag, (0, 1, 3, 2)) 
        
        # Inizializza il background per l'EMA
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # EMA
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # Flatten delle coordinate e concatenazione con la mask (12 valori totali)
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    # Uniamo tutte le liste in due tensori Numpy
    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)

    return X, Y

# ==============================================================================
# 2. SPLIT E ESECUZIONE (NORMALIZZAZIONE RIGOROSA)
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)

print("\n--- CALCOLO STATISTICHE E NORMALIZZAZIONE ---")
# LE STATISTICHE SI CALCOLANO SOLO SUL TRAIN!
train_mean = np.mean(X_train_raw)
train_std = np.std(X_train_raw)

print(f"!!! VALORI DA SALVARE E HARDCODARE NEL TUO CODE.PY !!!")
print(f"MEAN: {train_mean:.6f}")
print(f"STD:  {train_std:.6f}")

# Applichiamo la stessa normalizzazione a entrambi
X_train = (X_train_raw - train_mean) / (train_std + 1e-7)
X_val = (X_val_raw - train_mean) / (train_std + 1e-7)

# Liberiamo memoria RAM
del X_train_raw
del X_val_raw


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato.
File 2/18 processato.
File 3/18 processato.
File 4/18 processato.
File 5/18 processato.
File 6/18 processato.
File 7/18 processato.
File 8/18 processato.
File 9/18 processato.
File 10/18 processato.
File 11/18 processato.
File 12/18 processato.
File 13/18 processato.
File 14/18 processato.
File 15/18 processato.
File 16/18 processato.
File 17/18 processato.
File 18/18 processato.

--- PREPARAZIONE VALIDATION SET ---
Inizio caricamento ed EMA Decluttering di 6 file...
File 1/6 processato.
File 2/6 processato.
File 3/6 processato.
File 4/6 processato.
File 5/6 processato.
File 6/6 processato.

--- CALCOLO STATISTICHE E NORMALIZZAZIONE ---
!!! VALORI DA SALVARE E HARDCODARE NEL TUO CODE.PY !!!
MEAN: 13.715407
STD:  45.612080


In [6]:
import itertools

PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = 2 * coords_cost_norm + mask_cost_norm 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = 2 * coords_cost_norm + mask_cost_norm
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [5]:
def embedded_summary(model, input_shape=(1, 120, 18), is_int8=False):
    total_params = model.count_params()
    
    # 1. Calcolo FLASH (4 byte per Float32, 1 byte per INT8)
    bytes_per_param = 1 if is_int8 else 4
    estimated_flash_kb = (total_params * bytes_per_param) / 1024
    
    # 2. Calcolo SRAM (Tensor Arena) con logica Adiacente (Buffer Reuse)
    bytes_per_activation = 1 if is_int8 else 4
    max_adjacent_ram_kb = 0
    
    # Memoria occupata dal layer precedente (inizializzata con la dimensione dell'input)
    previous_layer_size = (np.prod(input_shape) * bytes_per_activation) / 1024
    
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            # Per i layer come "Concatenate" che potrebbero avere output multipli/strani
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        current_layer_size = (num_elements * bytes_per_activation) / 1024
        
        # IL FIX È QUI: Sommiamo il layer precedente e il layer corrente!
        # È il momento esatto in cui TFLM consuma più RAM durante l'esecuzione di questo layer.
        current_peak = previous_layer_size + current_layer_size
        
        if current_peak > max_adjacent_ram_kb:
            max_adjacent_ram_kb = current_peak
            
        previous_layer_size = current_layer_size

    print("============================================")
    mode_str = "INT8 (Quantizzato)" if is_int8 else "FLOAT32 (Training)"
    print(f"   REPORT REQUISITI ESP32-S3 [{mode_str}]   ")
    print("============================================")
    print(f" Memoria FLASH stimata : ~{estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{max_adjacent_ram_kb:.2f} KB (Limite: 300 KB)")
    print("============================================\n")

In [7]:
# ==============================================================================
# 2. ARCHITETTURA EEAI-NET V8 "CORAZZATA" 
# ==============================================================================
def build_eeai_model_v8_corazzata(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(64, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(64, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 3))(x) 
    
    x = layers.Flatten(name="flatten_spatial_map")(x) 
    
    x = layers.Dense(256, activation='relu', name="features_deep_1")(x)
    x = layers.Dropout(0.3, name="drop_features")(x) 
    common_feat = layers.Dense(128, activation='relu', name="features_deep_2")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    # UNIONE DELLE TESTE PER L'HUNGARIAN MATCHING
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V8_Corazzata")

model_v8_final = build_eeai_model_v8_corazzata()

if 'embedded_summary' in globals():
    embedded_summary(model_v8_final)  # Torneremo ai nostri sicuri ~684 KB!

# ==============================================================================
# 3. FINE TUNING: OTTIMIZZATORE E CALLBACKS
# ==============================================================================


# Learning rate dimezzato in partenza (0.0005) per una discesa chirurgica
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model_v8_final.compile(
    optimizer=optimizer,
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

# Pazienza aumentata (5 e 15) per cuocere la rete a fuoco lento
checkpoint_v8 = ModelCheckpoint("eeai_best_model_v8.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)

print("\n--- INIZIO ADDESTRAMENTO V8 CORAZZATA (MINI-BATCH GRADIENT DESCENT) ---")
history_v8_final = model_v8_final.fit(
    X_train, Y_train,                # Usiamo i tensori in RAM, non il generatore!
    validation_data=(X_val, Y_val),  # Usiamo i tensori in RAM!
    batch_size=32,                   # LA MAGIA AVVIENE QUI: 32 frame alla volta
    shuffle=True,                    # Rimescolamento perfetto per evitare bias
    epochs=300,
    callbacks=[checkpoint_v8, reduce_lr, early_stop], 
    verbose=1
)

   REPORT REQUISITI ESP32-S3 [FLOAT32 (Training)]   
 Memoria FLASH stimata : ~683.92 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~0.00 KB (Limite: 300 KB)


--- INIZIO ADDESTRAMENTO V8 CORAZZATA (MINI-BATCH GRADIENT DESCENT) ---


W0000 00:00:1783069286.003738    5374 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.
W0000 00:00:1783069287.182857    5374 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.


Epoch 1/300


I0000 00:00:1783069289.075771    5862 service.cc:153] XLA service 0x5aed16b395c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1783069289.075788    5862 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1650, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.10.1)
I0000 00:00:1783069289.132459    5862 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1783069289.435588    5862 cuda_dnn.cc:461] Loaded cuDNN version 91001
I0000 00:00:1783069289.485373    5862 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3906__.63


  61/4219 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - hungarian_mask_acc: 0.5734 - hungarian_rmse_metres: 2.4090 - loss: 17.2967

...

4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - hungarian_mask_acc: 0.7485 - hungarian_rmse_metres: 0.8312 - loss: 2.7266

I0000 00:00:1783069303.932172    5863 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3906__.63


4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - hungarian_mask_acc: 0.7486 - hungarian_rmse_metres: 0.8310 - loss: 2.7255

W0000 00:00:1783069307.615831    5374 cpu_allocator_impl.cc:82] Allocation of 388800000 exceeds 10% of free system memory.
W0000 00:00:1783069307.993323    5374 cpu_allocator_impl.cc:82] Allocation of 388800000 exceeds 10% of free system memory.
I0000 00:00:1783069308.447554    5863 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_25568__.24
I0000 00:00:1783069311.074846    5859 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_25568__.24



Epoch 1: val_loss improved from None to 0.98342, saving model to eeai_best_model_v8.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 24s 4ms/step - hungarian_mask_acc: 0.8288 - hungarian_rmse_metres: 0.6519 - loss: 1.6356 - val_hungarian_mask_acc: 0.9173 - val_hungarian_rmse_metres: 0.4806 - val_loss: 0.9834 - learning_rate: 5.0000e-04
Epoch 2/300
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - hungarian_mask_acc: 0.9238 - hungarian_rmse_metres: 0.4921 - loss: 0.9057
Epoch 2: val_loss improved from 0.98342 to 0.95815, saving model to eeai_best_model_v8.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - hungarian_mask_acc: 0.9319 - hungarian_rmse_metres: 0.4675 - loss: 0.8237 - val_hungarian_mask_acc: 0.8946 - val_hungarian_rmse_metres: 0.4622 - val_loss: 0.9582 - learning_rate: 5.0000e-04
Epoch 3/300
4210/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - hungarian_mask_acc: 0.9484 - hungarian_rmse_metres: 0.4147 - loss: 0.6553
Epoch 3: val_loss improved from 0.95815 to 0.72060, saving model to eeai_best_mode

In [10]:
# ==============================================================================
# 2. ARCHITETTURA EEAI-NET V10 "CONDIVISA" 
# ==============================================================================

def build_eeai_model_v10_shared(n_radars=6, n_antennas=3, n_bins=120):
    # L'input ora ha forma (6, 120, 3)
    inputs = layers.Input(shape=(n_radars, n_bins, n_antennas), name="radar_input")

    # --- FEATURE EXTRACTOR CONDIVISO ---
    # Kernel (1, 5): scorre sui 120 bin ma NON mescola i 6 radar (altezza kernel = 1)
    x = layers.SeparableConv2D(32, (1, 5), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((1, 2))(x)  # Shape diventa: (6, 60, 32)
    
    # Allarghiamo il campo recettivo sui bin (Dilation) mantenendo bassi i parametri
    x = layers.SeparableConv2D(64, (1, 3), padding='same', dilation_rate=(1, 2), activation='relu')(x)
    x = layers.MaxPooling2D((1, 2))(x)  # Shape diventa: (6, 30, 64)
    
    # SQUEEZE LAYER: Abbattiamo i canali prima del Flatten per salvare memoria!
    x = layers.Conv2D(16, (1, 1), padding='same', activation='relu')(x) # Shape: (6, 30, 16)
    
    # --- FUSIONE GEOMETRICA ---
    x = layers.Flatten(name="flatten_spatial_map")(x)  # 6 * 30 * 16 = 2880 feature
    x = layers.Dropout(0.4, name="heavy_spatial_dropout")(x) # Regolarizzazione dura
    
    x = layers.Dense(128, activation='relu', name="features_deep_1")(x)
    x = layers.Dropout(0.2, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features_deep_2")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V10_Shared")

model_v10_final = build_eeai_model_v10_shared()

if 'embedded_summary' in globals():
    embedded_summary(model_v10_final) # Vedrai Flash e SRAM crollare a livelli ottimali!

# ==============================================================================
# 3. GENERATORE CON RADAR DROPOUT E FINE TUNING
# ==============================================================================

class RadarAugmentGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, Y, batch_size=32, drop_prob=0.35, max_drop=2):
        self.X = X
        self.Y = Y
        self.batch_size = batch_size
        self.drop_prob = drop_prob   # 35% di probabilità di oscurare dei radar
        self.max_drop = max_drop     # Massimo 2 radar oscurati contemporaneamente
        self.indices = np.arange(len(self.X))
        np.random.shuffle(self.indices)
        
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
        
    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx+1) * self.batch_size]
        X_batch = self.X[batch_idx].copy() # Importante il .copy() per non modificare la RAM
        Y_batch = self.Y[batch_idx]
        
        # Applica il Radar Dropout on-the-fly
        for i in range(len(X_batch)):
            if np.random.rand() < self.drop_prob:
                num_drop = np.random.randint(1, self.max_drop + 1)
                drop_idx = np.random.choice(6, num_drop, replace=False)
                X_batch[i, drop_idx, :, :] = 0.0 # Azzera completamente i radar scelti
                
        return X_batch, Y_batch
        
    def on_epoch_end(self):
        np.random.shuffle(self.indices)

# Inizializziamo il generatore SOLO SUL TRAIN
train_generator = RadarAugmentGenerator(X_train, Y_train, batch_size=32)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model_v10_final.compile(
    optimizer=optimizer,
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_v10 = ModelCheckpoint("eeai_best_model_v10.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)

print("\n--- INIZIO ADDESTRAMENTO V10 CONDIVISA CON RADAR DROPOUT ---")
history_v10_final = model_v10_final.fit(
    train_generator,                 # LA NUOVA MAGIA AVVIENE QUI: Dati aumentati!
    validation_data=(X_val, Y_val),  # Il Validation Set rimane pulito, SENZA dropout!
    epochs=300,
    callbacks=[checkpoint_v10, reduce_lr, early_stop], 
    verbose=1
)

   REPORT REQUISITI ESP32-S3 [FLOAT32 (Training)]   
 Memoria FLASH stimata : ~1489.04 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~0.00 KB (Limite: 300 KB)


--- INIZIO ADDESTRAMENTO V10 CONDIVISA CON RADAR DROPOUT ---
Epoch 1/300


I0000 00:00:1783070748.915648    5862 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2325304__.47


2660/4219 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - hungarian_mask_acc: 0.7011 - hungarian_rmse_metres: 0.9452 - loss: 3.6328

I0000 00:00:1783070760.171728    5861 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2325304__.47


4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - hungarian_mask_acc: 0.7320 - hungarian_rmse_metres: 0.8601 - loss: 3.0106

I0000 00:00:1783070768.378672    5862 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2346976__.20
I0000 00:00:1783070770.789263    5860 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2346976__.20



Epoch 1: val_loss improved from None to 1.18317, saving model to eeai_best_model_v10.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 24s 5ms/step - hungarian_mask_acc: 0.7987 - hungarian_rmse_metres: 0.6925 - loss: 1.8300 - val_hungarian_mask_acc: 0.8436 - val_hungarian_rmse_metres: 0.5192 - val_loss: 1.1832 - learning_rate: 5.0000e-04
Epoch 2/300
4210/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - hungarian_mask_acc: 0.8754 - hungarian_rmse_metres: 0.5670 - loss: 1.2167
Epoch 2: val_loss improved from 1.18317 to 0.96844, saving model to eeai_best_model_v10.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - hungarian_mask_acc: 0.8865 - hungarian_rmse_metres: 0.5468 - loss: 1.1418 - val_hungarian_mask_acc: 0.9106 - val_hungarian_rmse_metres: 0.4685 - val_loss: 0.9684 - learning_rate: 5.0000e-04
Epoch 3/300
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - hungarian_mask_acc: 0.9096 - hungarian_rmse_metres: 0.4943 - loss: 0.9571
Epoch 3: val_loss did not improve from 0.96844
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 15

In [9]:
def build_eeai_model_v11_senza_freni(n_radars=6, n_antennas=3, n_bins=120):
    inputs = layers.Input(shape=(n_radars, n_bins, n_antennas), name="radar_input")

    # --- FEATURE EXTRACTOR CONDIVISO ---
    x = layers.SeparableConv2D(32, (1, 5), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((1, 2))(x)  # Shape: (6, 60, 32) -> RISOLUZIONE 12 CM!
    
    # Dilation per guardare lontano, ma SENZA IL SECONDO POOLING!
    x = layers.SeparableConv2D(64, (1, 3), padding='same', dilation_rate=(1, 2), activation='relu')(x)
    # Shape resta: (6, 60, 64)
    
    # SQUEEZE LAYER A 8 CANALI (Compensa il raddoppio dei bin per salvare la Flash)
    x = layers.Conv2D(16, (1, 1), padding='same', activation='relu')(x) # Shape: (6, 60, 8)
    
    # --- FUSIONE GEOMETRICA ---
    # 6 * 60 * 16 = 5760 features
    x = layers.Flatten(name="flatten_spatial_map")(x)  
    x = layers.Dropout(0.3, name="heavy_spatial_dropout")(x) # Leggermente allentato per non bloccare il Train
    
    # Manteniamo il tuo Dense massiccio!
    x = layers.Dense(128, activation='relu', name="features_deep_1")(x)
    x = layers.Dropout(0.2, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features_deep_2")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V11_SenzaFreni")

model_v11_final = build_eeai_model_v11_senza_freni()

if 'embedded_summary' in globals():
    embedded_summary(model_v11_final) # Vedrai Flash e SRAM crollare a livelli ottimali!

# ==============================================================================
# 3. GENERATORE CON RADAR DROPOUT E FINE TUNING
# ==============================================================================

class RadarAugmentGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, Y, batch_size=32, drop_prob=0.35, max_drop=2):
        self.X = X
        self.Y = Y
        self.batch_size = batch_size
        self.drop_prob = drop_prob   # 35% di probabilità di oscurare dei radar
        self.max_drop = max_drop     # Massimo 2 radar oscurati contemporaneamente
        self.indices = np.arange(len(self.X))
        np.random.shuffle(self.indices)
        
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
        
    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx+1) * self.batch_size]
        X_batch = self.X[batch_idx].copy() # Importante il .copy() per non modificare la RAM
        Y_batch = self.Y[batch_idx]
        
        # Applica il Radar Dropout on-the-fly
        for i in range(len(X_batch)):
            if np.random.rand() < self.drop_prob:
                num_drop = np.random.randint(1, self.max_drop + 1)
                drop_idx = np.random.choice(6, num_drop, replace=False)
                X_batch[i, drop_idx, :, :] = 0.0 # Azzera completamente i radar scelti
                
        return X_batch, Y_batch
        
    def on_epoch_end(self):
        np.random.shuffle(self.indices)

# Inizializziamo il generatore SOLO SUL TRAIN
train_generator = RadarAugmentGenerator(X_train, Y_train, batch_size=32)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model_v11_final.compile(
    optimizer=optimizer,
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_v11 = ModelCheckpoint("eeai_best_model_v11.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)

print("\n--- INIZIO ADDESTRAMENTO V11 CONDIVISA CON RADAR DROPOUT ---")
history_v11_final = model_v11_final.fit(
    train_generator,                 # LA NUOVA MAGIA AVVIENE QUI: Dati aumentati!
    validation_data=(X_val, Y_val),  # Il Validation Set rimane pulito, SENZA dropout!
    epochs=300,
    callbacks=[checkpoint_v11, reduce_lr, early_stop], 
    verbose=1
)

   REPORT REQUISITI ESP32-S3 [FLOAT32 (Training)]   
 Memoria FLASH stimata : ~2929.04 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~0.00 KB (Limite: 300 KB)


--- INIZIO ADDESTRAMENTO V11 CONDIVISA CON RADAR DROPOUT ---
Epoch 1/300


/home/marco/yes/envs/edge_ai_env/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1783070091.100968    5374 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1783070092.833361    5859 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1192703__.39


3457/4219 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - hungarian_mask_acc: 0.7115 - hungarian_rmse_metres: 0.8637 - loss: 2.9650

I0000 00:00:1783070107.376416    5860 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1192703__.39


4203/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - hungarian_mask_acc: 0.7261 - hungarian_rmse_metres: 0.8318 - loss: 2.7560

W0000 00:00:1783070112.808522    5374 cpu_allocator_impl.cc:82] Allocation of 388800000 exceeds 10% of free system memory.
I0000 00:00:1783070113.638483    5862 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1214374__.18
I0000 00:00:1783070116.105473    5862 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1214374__.18



Epoch 1: val_loss improved from None to 1.09509, saving model to eeai_best_model_v11.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 26s 5ms/step - hungarian_mask_acc: 0.8010 - hungarian_rmse_metres: 0.6734 - loss: 1.7352 - val_hungarian_mask_acc: 0.8584 - val_hungarian_rmse_metres: 0.4975 - val_loss: 1.0951 - learning_rate: 5.0000e-04
Epoch 2/300
4207/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - hungarian_mask_acc: 0.8884 - hungarian_rmse_metres: 0.5426 - loss: 1.1267
Epoch 2: val_loss improved from 1.09509 to 1.00354, saving model to eeai_best_model_v11.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - hungarian_mask_acc: 0.8993 - hungarian_rmse_metres: 0.5220 - loss: 1.0506 - val_hungarian_mask_acc: 0.8786 - val_hungarian_rmse_metres: 0.4794 - val_loss: 1.0035 - learning_rate: 5.0000e-04
Epoch 3/300
4205/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - hungarian_mask_acc: 0.9236 - hungarian_rmse_metres: 0.4765 - loss: 0.8814
Epoch 3: val_loss improved from 1.00354 to 0.87283, saving model to eeai_best_mo

Model: "EEAI_Net_V10_Wide"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ radar_input         │ (None, 1, 120,    │          0 │ -                 │
│ (InputLayer)        │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 1, 120,    │      5,824 │ radar_input[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 1, 60, 64) │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_6  │ (None, 1, 60,     │      8,512 │ max_pooling2d_8[… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 60,     │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 1, 30,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_7  │ (None, 1, 30,     │     33,408 │ max_pooling2d_9[… │
│ (SeparableConv2D)   │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 30,     │      1,024 │ separable_conv2d… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, 1, 15,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_8  │ (None, 1, 15,     │     33,664 │ max_pooling2d_10… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 15,     │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_11    │ (None, 1, 5, 128) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_spatial_map │ (None, 640)       │          0 │ max_pooling2d_11… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features_deep_1     │ (None, 128)       │     82,048 │ flatten_spatial_… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_features       │ (None, 128)       │          0 │ features_deep_1[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features_deep_2     │ (None, 64)        │      8,256 │ drop_features[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coords_head (Dense) │ (None, 8)         │        520 │ features_deep_2[

 Total params: 174,540 (681.80 KB)

 Trainable params: 173,516 (677.80 KB)

 Non-trainable params: 1,024 (4.00 KB)

...

4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - hungarian_mask_acc: 0.8400 - hungarian_rmse_metres: 1.0089 - loss: 2.2923

I0000 00:00:1782729542.543763    5429 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8648__.81


4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.8401 - hungarian_rmse_metres: 1.0086 - loss: 2.2909

I0000 00:00:1782729551.664809    5429 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_30439__.24
I0000 00:00:1782729554.855440    5428 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_30439__.24



Epoch 1: val_loss improved from None to 1.28004, saving model to eeai_best_model_v10.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - hungarian_mask_acc: 0.8973 - hungarian_rmse_metres: 0.8174 - loss: 1.4747 - val_hungarian_mask_acc: 0.9481 - val_hungarian_rmse_metres: 0.8080 - val_loss: 1.2800 - learning_rate: 5.0000e-04
Epoch 2/300
4213/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - hungarian_mask_acc: 0.9456 - hungarian_rmse_metres: 0.6105 - loss: 0.7844
Epoch 2: val_loss did not improve from 1.28004
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - hungarian_mask_acc: 0.9478 - hungarian_rmse_metres: 0.5848 - loss: 0.7286 - val_hungarian_mask_acc: 0.9190 - val_hungarian_rmse_metres: 0.8569 - val_loss: 1.5173 - learning_rate: 5.0000e-04
Epoch 3/300
4214/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - hungarian_mask_acc: 0.9530 - hungarian_rmse_metres: 0.5348 - loss: 0.6209
Epoch 3: val_loss did not improve from 1.28004
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - hungarian_mask_acc: 0.9543 - hunga

### Come leggere le Metriche della nostra Hungarian EEAI-Net (V7)

Con l'introduzione dell'**Hungarian Matching** (Permutation Invariant Training), le nostre due teste sono state fuse per calcolare un unico costo globale. Ecco i 4 valori fondamentali che vedrai scorrere sullo schermo e come interpretarli:

1. **hungarian_rmse_metres (L'Errore Spaziale Reale)**
È il traduttore fisico. Indica la distanza media in metri tra le tue predizioni (le X rosse) e le persone reali (i pallini verdi), misurata **dopo** che l'algoritmo ha trovato l'incrocio matematico perfetto. 
*Esempio:* Se vale `1.54`, stai sbagliando in media di 1 metro e mezzo. Più questo numero scende verso lo zero, più le X rosse "inseguiranno" fedelmente i bersagli veri, ignorando i fantasmi.

2. **loss (Il Voto Negativo Globale - Il motore dei gradienti)**
È il "costo" complessivo che la rete usa per correggere i propri errori sui dati di Addestramento. Unisce due punizioni: 
- L'errore balistico di posizione (MSE).
- La penalità se allucina "fantasmi" in slot vuoti (Binary Crossentropy moltiplicata per `2.5`). 
La loss viene calcolata *solo ed esclusivamente* sulla permutazione migliore delle 24 possibili. La rete cerca disperatamente di abbassare questo numero aggiornando i suoi pesi convoluzionali.

3. **val_loss (La Validation Loss - Il Re assoluto dell'addestramento)**
È lo stesso identico calcolo globale della `loss`, ma applicato ai dati che la rete **non ha mai visto** (l'esame di maturità a libro chiuso, es. le finestre 11 e 15). 
*Attenzione:* Questo è il numero più importante di tutti. L'`EarlyStopping` e il `ModelCheckpoint` guardano *esclusivamente* la `val_loss` per capire se il modello sta generalizzando la fisica del radar o se si sta solo imparando a memoria il training set (Overfitting). Finché scende, sei sulla strada giusta.

4. **val_hungarian_rmse_metres (La Prestazione Operativa - Il numero per la Tesi)**
È l'errore spaziale in metri misurato sui dati sconosciuti di validazione. Questo è il dato ufficiale che certificherà la bontà del tuo progetto. Nel tuo report o nella tesi scriverai: *"Il modello ha dimostrato un errore operativo reale di X metri su scenari mai visti durante l'addestramento"*.

Epoch 11/300
4212/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - hungarian_mask_acc: 0.9730 - hungarian_rmse_metres: 0.3839 - loss: 0.3557
Epoch 11: val_loss improved from 0.55936 to 0.54816, saving model to eeai_best_model_v8_2.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - hungarian_mask_acc: 0.9734 - hungarian_rmse_metres: 0.3841 - loss: 0.3555 - val_hungarian_mask_acc: 0.9619 - val_hungarian_rmse_metres: 0.4560 - val_loss: 0.5482 - learning_rate: 5.0000e-04

In [11]:
# ==============================================================================
# VISUALIZZATORE 3.1 (Fix Output 12 Dimensioni V8)
# ==============================================================================

file_target = "dataset/data/window_000011.npz"

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V8)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.002
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)
    
    print("Applicazione Standardizzazione Globale...")
    # ATTENZIONE: Sostituisci questi due zeri con i veri numeri stampati dalla Cella 9!
    MEAN_TRAINING = 13.715407
    STD_TRAINING = 45.612080

    decluttered = (decluttered - MEAN_TRAINING) / (STD_TRAINING + 1e-7)


    print("Caricamento dei pesi migliori dal file .keras ...")
    
    # Caricamento del modello 
    model_v10_final = load_model(
        "eeai_best_model_v8.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_v10_final.predict(decluttered, verbose=0)
    
    # ==========================================
    # IL FIX E' QUI: Slicing corretto per la V10
    # ==========================================
    # preds ha dimensione [T, 12]. 
    # Prendiamo le prime 8 colonne per le coordinate, e le ultime 4 per le maschere.
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # Titolo aggiornato
            ax.set_title(f"Radar V8 | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (V8)...
Applicazione Standardizzazione Globale...
Caricamento dei pesi migliori dal file .keras ...


I0000 00:00:1783072184.253879    5858 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_4864372__.9
I0000 00:00:1783072184.699761    5862 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_4864962__.9


Dati pronti! Inizializzazione Radar...


In [ ]:
# ==============================================================================
# VISUALIZZATORE 4.0 (Compatibile con V10/V11 "Condivise" Shape: 6, 120, 3)
# ==============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from tensorflow.keras.models import load_model

file_target = "dataset/data/window_000011.npz"
# Seleziona qui il nome del modello che vuoi testare
NOME_MODELLO = "eeai_best_model_v11.keras" 

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print(f"Elaborazione filtri e previsioni in corso ({NOME_MODELLO})...")
    
    # 1. Calcolo Magnitudo: (T, 6, 3, 120)
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
    
    # ==========================================
    # FIX GEOMETRICO: Trasposizione per V10/V11
    # ==========================================
    # Da (T, 6, 3, 120) a (T, 6, 120, 3)
    mag_reshaped = np.transpose(mag, (0, 1, 3, 2))
    
    decluttered = np.zeros_like(mag_reshaped)
    bg = np.copy(mag_reshaped[0])
    alpha = 0.002
    
    for t in range(T):
        bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag_reshaped[t] - bg)
    
    print("Applicazione Standardizzazione Globale...")
    # ATTENZIONE: Assicurati che questi siano i valori del training su cui ha imparato il modello!
    MEAN_TRAINING = 13.715407
    STD_TRAINING = 45.612080

    decluttered = (decluttered - MEAN_TRAINING) / (STD_TRAINING + 1e-7)

    print(f"Caricamento dei pesi migliori dal file {NOME_MODELLO} ...")
    
    # Caricamento del modello 
    model_final = load_model(
        NOME_MODELLO,
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_final.predict(decluttered, verbose=0)
    
    # Slicing invariato e corretto: prime 8 coord, ultime 4 maschere
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # Titolo aggiornato dinamicamente
            modello_str = NOME_MODELLO.split('.')[0]
            ax.set_title(f"Radar {modello_str.upper()} | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (eeai_best_model_v11.keras)...
Applicazione Standardizzazione Globale...
Caricamento dei pesi migliori dal file eeai_best_model_v11.keras ...


I0000 00:00:1783072662.464658    5862 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_4869149__.3
I0000 00:00:1783072662.931191    5858 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_4869746__.3


Dati pronti! Inizializzazione Radar...
